# Token-level T5 FactorVAE — structured library driver

This notebook is now a thin experiment driver. The reusable code lives under `src/emotion_latent_learning/` and is grouped into `config`, `data`, `schemas`, `modeling`, `training`, `evaluation`, and `utils` subpackages.


## Import the local library

When running from this repository folder, the cell below makes `src/` importable without requiring an editable install. For normal project use, run `pip install -e .` once from the project root.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
SRC = PROJECT_ROOT / "src"
if SRC.exists() and str(SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC.resolve()))

from emotion_latent_learning import *


## Configuration

The defaults match the original notebook. Override values here before data/model construction when running ablations.


In [2]:
DATA_CONFIG, PROMPT_CONFIG, MODEL_CONFIG, LOSS_CONFIG, SCHEDULE_CONFIG, EXPERIMENT_CONFIG = default_configs()

# Example quick-test overrides. Uncomment when you want a short smoke run.
DATA_CONFIG.train_batch_size = 32
DATA_CONFIG.eval_batch_size = 64
# SCHEDULE_CONFIG.num_epochs = 1
# EXPERIMENT_CONFIG.output_dir = "factorvae_smoke_artifacts"

set_seed(EXPERIMENT_CONFIG.seed)
device = get_device()
print("device:", device)
print("seed:", EXPERIMENT_CONFIG.seed)


device: cuda
seed: 42


## Optional: switch to SemEval-2018 Task 1 EI-reg

Uncomment the settings below to train on SemEval emotion-intensity regression instead of GoEmotions.

The library now tries to download or extract the dataset automatically when `semeval_data_dir` is missing. If public mirrors fail, download the official archive manually and set `DATA_CONFIG.semeval_download_url` to the local `.zip` path.


In [ ]:
DATA_CONFIG.dataset_name = "semval2018_ei_reg"
DATA_CONFIG.semeval_data_dir = "../data/SemEval2018-Task1"
DATA_CONFIG.semeval_auto_download = True
DATA_CONFIG.semeval_download_url = "https://saifmohammad.com/WebDocs/AIT-2018/AIT2018-DATA/SemEval2018-Task1-all-data.zip"  # or "~/Downloads/SemEval2018-Task1.zip"
DATA_CONFIG.semeval_language = "En"
DATA_CONFIG.semeval_emotions = ("anger", "fear", "joy", "sadness")
MODEL_CONFIG.num_scalar_factors = len(DATA_CONFIG.semeval_emotions)
EXPERIMENT_CONFIG.output_dir = "factorvae_semval2018_ei_reg_artifacts"

# Manual one-shot preparation, useful when you have the official archive locally:
download_semval2018_ei_reg(
    DATA_CONFIG.semeval_data_dir,
    url=DATA_CONFIG.semeval_download_url,
    language=DATA_CONFIG.semeval_language,
    emotions=DATA_CONFIG.semeval_emotions,
)


PosixPath('../data/SemEval2018-Task1')

## Data loading

The library loads GoEmotions, creates train/threshold/validation/test loaders, computes positive-class weights, and fixes qualitative monitor examples.


In [3]:
data = build_data_bundle(
    data_config=DATA_CONFIG,
    model_config=MODEL_CONFIG,
    experiment_config=EXPERIMENT_CONFIG,
)

print("dataset repo:", DATA_CONFIG.dataset_repo)
print("dataset config:", DATA_CONFIG.dataset_config)
print("num labels:", data.num_labels)
print("first five labels:", list(data.emotion_names)[:5])
print("train core samples:", len(data.train_core_dataset))
print("threshold-tune samples:", len(data.threshold_tune_dataset))
print("train batches:", len(data.train_loader))
print("threshold batches:", len(data.threshold_loader))
print("val batches:", len(data.val_loader))
print("test batches:", len(data.test_loader))
print(
    "pos_weight stats:",
    {
        "min": round(float(data.pos_weight.min().item()), 3),
        "mean": round(float(data.pos_weight.mean().item()), 3),
        "max": round(float(data.pos_weight.max().item()), 3),
    },
)
print("monitor example indices:", data.monitor_example_indices)
for idx, target_labels in zip(data.monitor_example_indices, data.monitor_target_labels):
    preview = data.val_dataset[idx]["text"]
    print(f"- val index {idx}: edit targets={target_labels} | text={preview[:90]!r}")


dataset repo: google-research-datasets/go_emotions
dataset config: simplified
num labels: 28
first five labels: ['admiration', 'amusement', 'anger', 'annoyance', 'approval']
train core samples: 41240
threshold-tune samples: 2170
train batches: 1288
threshold batches: 34
val batches: 85
test batches: 85
pos_weight stats: {'min': 2.053, 'mean': 18.339, 'max': 20.0}
monitor example indices: [547, 2820]
- val index 547: edit targets=['excitement', 'annoyance'] | text='Meh good introduction. Sadly I am a pro philosopher so know all of this'
- val index 2820: edit targets=['disapproval', 'remorse'] | text="if it shatters your little ego i'm fine with this. :)"


## Build the experiment runtime

This creates the T5-FactorVAE model, optional FactorVAE discriminator, adversaries, optimizers, scheduler, context, state object, and output directory.


In [4]:
runtime = build_runtime(
    data=data,
    data_config=DATA_CONFIG,
    prompt_config=PROMPT_CONFIG,
    model_config=MODEL_CONFIG,
    loss_config=LOSS_CONFIG,
    schedule_config=SCHEDULE_CONFIG,
    experiment_config=EXPERIMENT_CONFIG,
    device=device,
)

summary = runtime_summary(runtime)
for key, value in summary.items():
    print(f"{key}: {value}")


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


device: cuda
hidden_size: 512
latent_dim: 284
num_labels: 28
dataset_name: goemotions
target_kind: multilabel
train_core_samples: 41240
threshold_tune_samples: 2170
train_batches: 1288
threshold_batches: 34
val_batches: 85
test_batches: 85
trainable_parameter_groups: 154
first_trainable_names: ['vae_encoder.net.0.weight', 'vae_encoder.net.0.bias', 'vae_encoder.net.3.weight', 'vae_encoder.net.3.bias', 'vae_encoder.net.6.weight', 'vae_encoder.net.6.bias', 'vae_encoder.mu.weight', 'vae_encoder.mu.bias', 'vae_encoder.logvar.weight', 'vae_encoder.logvar.bias']
output_dir: /mnt/disk1/Projects/NLP-Latent-Learning/factorvae_tokenlevel_clean_artifacts
lora_target_modules: ['q', 'v']
gpu_total_gb: 3.681884765625


## Train

The full epoch loop now lives in `run_training`. It handles LoRA scheduling, loss weights, calibration-threshold tuning, validation, history saving, qualitative monitoring, and checkpoints.


In [5]:
state = run_training(runtime, monitor=True, save_each_epoch=True)



[epoch 1] weights={'cls': 4.00, 'recon': 2.00, 'kl': 0.0000, 'tc': 0.0000, 'copy': 2.0000, 'vec_adv': 0.1000, 'res_adv': 0.2000, 'sep': 0.0100, 'orth': 0.0100, 'transfer': 0.2500, 'edit': 0.1000, 'res_scale': 1.0000} lora_enabled=False tc_mode=per_token threshold_mode=per_label


epoch 1/30 [train]:   0%|          | 0/1288 [00:00<?, ?it/s]

[epoch 1] train loss=12.8556 raw(recon=0.0165, kl=0.4963, cls=1.0317, tc=0.0000, copy=3.9628, vec_adv=0.9286, res_adv=0.9225, sep=0.0000, orth=0.0000, transfer=1.9295) weighted(recon=0.0329, kl=0.0000, cls=4.1270, tc=0.0000, copy=7.9257, vec_adv=0.0929, res_adv=0.1845, sep=0.0000, orth=0.0000, transfer=0.4824) threshold=per_label(mean=0.500, min=0.500, max=0.500) micro_f1=0.0997 macro_f1=0.0343 weighted_f1=0.1406 subset_acc=0.0000 hamming_acc=0.7080 jaccard_micro=0.0525 ap_micro=0.0488 lrap=0.1573 semval_pearson=0.0070 factor_dci=0.0000 branch_dci=0.0000 branch_mig=0.0000 emo_share=0.0000 leak_share=0.0000 split_r2=0.0000


epoch 1/30 [threshold]:   0%|          | 0/34 [00:00<?, ?it/s]

[epoch 1] calib loss=10.9374 raw(recon=0.0142, kl=1.1226, cls=1.0045, tc=0.0000, copy=3.0739, vec_adv=0.9069, res_adv=0.9236, sep=0.0000, orth=0.0000, transfer=1.8273) weighted(recon=0.0284, kl=0.0000, cls=4.0180, tc=0.0000, copy=6.1478, vec_adv=0.0907, res_adv=0.1847, sep=0.0000, orth=0.0000, transfer=0.4568) threshold=per_label(mean=0.341, min=0.010, max=0.539) micro_f1=0.0928 macro_f1=0.0964 weighted_f1=0.2254 subset_acc=0.0000 hamming_acc=0.3154 jaccard_micro=0.0486 ap_micro=0.0768 lrap=0.2199 semval_pearson=0.0316 factor_dci=0.0988 branch_dci=0.3246 branch_mig=-0.0689 emo_share=0.4470 leak_share=0.5411 split_r2=-0.0182


...

[epoch 6] calib loss=3.4829 raw(recon=0.0145, kl=5.7843, cls=0.4606, tc=0.0000, copy=0.0293, vec_adv=0.9260, res_adv=0.9284, sep=0.0000, orth=0.0000, transfer=0.4447) weighted(recon=0.0290, kl=0.0482, cls=1.8423, tc=0.0000, copy=0.0585, vec_adv=0.4630, res_adv=0.9284, sep=0.0000, orth=0.0000, transfer=0.1112) threshold=per_label(mean=0.711, min=0.226, max=0.951) micro_f1=0.5602 macro_f1=0.5147 weighted_f1=0.5691 subset_acc=0.3221 hamming_acc=0.9567 jaccard_micro=0.3891 ap_micro=0.5204 lrap=0.6831 semval_pearson=0.4551 factor_dci=0.1088 branch_dci=0.3247 branch_mig=0.1222 emo_share=0.5191 leak_share=0.4703 split_r2=0.0154


epoch 6/30 [val]:   0%|          | 0/85 [00:00<?, ?it/s]

[epoch 6] val   loss=3.4087 raw(recon=0.0144, kl=5.7357, cls=0.4470, tc=0.0000, copy=0.0290, vec_adv=0.9152, res_adv=0.9169, sep=0.0000, orth=0.0000, transfer=0.4360) weighted(recon=0.0288, kl=0.0478, cls=1.7881, tc=0.0000, copy=0.0581, vec_adv=0.4576, res_adv=0.9169, sep=0.0000, orth=0.0000, transfer=0.1090) threshold=per_label(mean=0.711, min=0.226, max=0.951) micro_f1=0.5549 macro_f1=0.4769 weighted_f1=0.5643 subset_acc=0.3319 hamming_acc=0.9558 jaccard_micro=0.3840 ap_micro=0.5507 lrap=0.7053 semval_pearson=0.4561 factor_dci=0.1230 branch_dci=0.3467 branch_mig=0.1648 emo_share=0.5689 leak_share=0.4239 split_r2=-0.0291

Epoch 6 qualitative monitor on fixed validation examples

------------------------------------------------------------------------------------------------------------------------
Example 1 | validation index=547
input text:
    Meh good introduction. Sadly I am a pro philosopher so know all of this
gold emotions: admiration
predicted emotions on input: sadness
loss s

epoch 7/30 [train]:   0%|          | 0/1288 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Compact training history view


In [ ]:
import pandas as pd

history_df = pd.DataFrame(runtime.state.history)
display(history_df.tail(12))


TypeError: 'module' object is not callable

## Final validation and test evaluation

This loads the best checkpoint, evaluates calibration/validation/test splits, saves `final_metrics.json`, and writes `final_checkpoint.pt`.


In [ ]:
final_metrics = final_evaluation(runtime, load_best=True)
print("\n[final] test classification report\n")
print(final_metrics["test"]["classification_report_text"])

print("\n[final] SemEval / continuous regression metrics")
for key in [
    "semeval_ei_reg_official_score",
    "semeval_ei_reg_pearson_macro",
    "semeval_ei_reg_pearson_micro",
    "semeval_ei_reg_pearson_high_gold_macro",
    "semeval_ei_reg_spearman_macro",
    "semeval_ei_reg_mae",
    "semeval_ei_reg_rmse",
]:
    print(f"{key}: {final_metrics['test'].get(key)}")

print("\n[final] FactorVAE factor-power metrics")
for key in [
    "factor_dci_disentanglement",
    "factor_dci_completeness",
    "factor_effective_num_factors",
    "factor_active_scalar_factors",
]:
    print(f"{key}: {final_metrics['test'].get(key)}")

print("\n[final] Emotion/meaning split metrics")
for key in [
    "emotion_in_scalar_r2",
    "emotion_leakage_vector_r2",
    "emotion_meaning_separation_r2",
    "emotion_meaning_separation_pearson",
    "scalar_vector_mean_abs_correlation",
]:
    print(f"{key}: {final_metrics['test'].get(key)}")


## Inspect factor power and emotion/meaning split

The aggregate metrics above are useful for tracking progress. These tables expose which scalar factors align with which target emotions and how much emotion leaks into the meaning branch.


In [ ]:
import pandas as pd

factor_power = final_metrics["test"].get("factor_power", {})
split_metrics = final_metrics["test"].get("emotion_meaning_split", {})

top_factor_rows = []
for emotion, row in factor_power.get("per_label_top_factor", {}).items():
    top_factor_rows.append({"emotion": emotion, **row})
display(pd.DataFrame(top_factor_rows))

split_summary = {
    "emotion_in_scalar_r2": split_metrics.get("emotion_in_scalar_r2"),
    "emotion_leakage_vector_r2": split_metrics.get("emotion_leakage_vector_r2"),
    "emotion_meaning_separation_r2": split_metrics.get("emotion_meaning_separation_r2"),
    "emotion_leakage_ratio_r2": split_metrics.get("emotion_leakage_ratio_r2"),
    "scalar_vector_mean_abs_correlation": split_metrics.get("scalar_vector_mean_abs_correlation"),
}
display(pd.DataFrame([split_summary]))


## Optional prompt experiment after training

This compares the prompt candidates from `EXPERIMENT_CONFIG.prompt_candidates` on a small validation subset using VAE memory.


In [ ]:
prompt_results = run_prompt_experiment(runtime)
prompt_results


## Final qualitative walkthrough on the best checkpoint


In [ ]:
run_final_walkthrough(runtime)


## Optional single-example latent editing demo


In [ ]:
edit_demo = single_example_latent_edit_demo(runtime)
edit_demo


## Ablation knobs retained on purpose

The key ablations are now configuration changes rather than notebook rewrites:

- `MODEL_CONFIG.latent_pool_heads`: attention-pooling head count.
- `MODEL_CONFIG.vector_latent_dim`: vector/semantic latent capacity.
- `MODEL_CONFIG.num_scalar_factors`: scalar emotion-factor count.
- `MODEL_CONFIG.attention_source`: `scalar_only`, `vector_only`, `latent_full`, or `encoder_sequence`.
- `MODEL_CONFIG.pooling_mode`: `per_scalar_dim` or `joint_scalar_vector`.
- `MODEL_CONFIG.classifier_mode`: `joint_mlp` or `per_emotion_mlp`.
- `MODEL_CONFIG.classifier_parameterization`: `standard` or `orthogonal`.
- `MODEL_CONFIG.use_skip_connection`: add/remove residual memory path.
- `LOSS_CONFIG.tc_weight` and `LOSS_CONFIG.tc_subspace`: enable/disable FactorVAE total-correlation pressure.
- `LOSS_CONFIG.vector_adv_weight` and `LOSS_CONFIG.residual_adv_weight`: enable/disable adversarial leakage controls.
- `SCHEDULE_CONFIG.lora_start_epoch` and `SCHEDULE_CONFIG.copy_loss_start_epoch`: vary when decoder-copy adaptation begins.
- `PROMPT_CONFIG.use_prompt`, `PROMPT_CONFIG.prompt_text`, and `PROMPT_CONFIG.mask_prompt_loss`: decoder prompt/copy-path variants.
